# 🛒 AisleIQ — Custom YOLOv8 Grocery Model Training

**This notebook trains a YOLOv8 object detection model using a public grocery dataset from Roboflow Universe.**

### Steps:
1. Install dependencies
2. Connect to Roboflow and download grocery dataset
3. Train YOLOv8 model
4. Evaluate the model
5. Download `best.pt` for use in the AisleIQ vision system

> 💡 **Run this on Google Colab with a free GPU:**  
> `Runtime → Change runtime type → T4 GPU`

## 1. Install Dependencies

In [ ]:
!pip install ultralytics roboflow --quiet
import ultralytics
ultralytics.checks()

## 2. Download Dataset from Roboflow

We are using the **'Grocery Store'** dataset from Roboflow Universe.  
It contains annotated images of common grocery/supermarket products.

🔗 Dataset: https://universe.roboflow.com/grocery-prbmk/grocery-store-obksk

> **To use your own dataset:**
> 1. Go to https://universe.roboflow.com and find a dataset you like
> 2. Click 'Download Dataset' → Format: YOLOv8 → 'Show download code'
> 3. Replace the `rf.workspace(...)` call below with the code snippet provided

In [ ]:
from roboflow import Roboflow

# ⚠️ Replace this with your own Roboflow API key
# Get it for free at: https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# --- OPTION A: Grocery Store dataset (general grocery items) ---
project = rf.workspace("grocery-prbmk").project("grocery-store-obksk")

# --- OPTION B: Retail Products Detection (more SKU-focused) ---
# project = rf.workspace("roboflow-universe-projects").project("grocery-dataset-yxgzs")

version = project.version(1)
dataset = version.download("yolov8")
print(f"Dataset downloaded to: {dataset.location}")
print(f"Classes: {dataset.classes}")

## 3. Train the YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Load a small, fast pre-trained model (yolov8n = nano)
# For better accuracy, use yolov8s.pt (small) or yolov8m.pt (medium)
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,          # Increase to 100 for better accuracy
    imgsz=640,          # Standard image size
    batch=16,           # Reduce to 8 if you get out-of-memory errors
    name='aisleiq_grocery',
    patience=10,        # Stop early if model stops improving
    device=0,           # 0 = GPU (if available), 'cpu' = CPU
    plots=True          # Generate training plots
)

print("\n✅ Training complete!")
print(f"Best model saved at: {results.save_dir}/weights/best.pt")

## 4. Evaluate the Model

In [ ]:
# Validate the trained model on the test set
model_path = f"{results.save_dir}/weights/best.pt"
trained_model = YOLO(model_path)
metrics = trained_model.val(data=f"{dataset.location}/data.yaml")

print(f"\n📊 Validation Results:")
print(f"  mAP50:      {metrics.box.map50:.3f}")
print(f"  mAP50-95:   {metrics.box.map:.3f}")
print(f"  Precision:  {metrics.box.p.mean():.3f}")
print(f"  Recall:     {metrics.box.r.mean():.3f}")

## 5. Inspect Class Labels

Run this to get the exact class names your model recognizes. You'll need these to map labels → SKUs in `api_client.py`.

In [ ]:
import yaml, os

data_yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

class_names = data.get('names', [])
print("✅ Classes detected by your model:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

print("\n💡 Copy this list and update api_client.py with matching SKU IDs and prices!")

## 6. Download Your Model

Run the cell below to download `best.pt` to your computer.

In [ ]:
from google.colab import files
import shutil

best_pt = f"{results.save_dir}/weights/best.pt"
shutil.copy(best_pt, '/content/aisleiq_best.pt')
files.download('/content/aisleiq_best.pt')

print("⬇️ Downloading aisleiq_best.pt ...")
print("\nNext steps:")
print("  1. Move aisleiq_best.pt to your AisleIQ/vision/ folder")
print("  2. Update vision/main.py: model_path='aisleiq_best.pt'")
print("  3. Update vision/api_client.py with the class names above")